In [2]:
"""
Day 2 - Sales Analytics Without Pandas
Squad Python Specialization - Data Science, AI & Analytics

Tasks 1-7 use only the standard library. pandas is allowed in task 8, to check your work.
"""
import csv
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"
OUTPUT_PATH = Path("output") / "sales_report.txt"

In [15]:
def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    with path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
    print(f"Rows read: {len(rows)}")
    print(f"Columns found: {', '.join(reader.fieldnames or [])}")
    return rows


read_sales()

Rows read: 50
Columns found: sale_id, sale_date, store, product, quantity, unit_price_kes, payment_method


[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3004',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'Xiaomi Redmi Buds',
  'quantity': '2',
  'unit_price_kes': '3199',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3005',
  'sale_date': '2026-09-03',
  'store': 'Nakuru Westside',
  'product': 'Tecno Spark 20',
  'quantity': '2',
  'unit_price_kes': '13499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3006',
  'sale_date': '2026-09-03',
  's

In [16]:
def parse_number(value: str) -> float | None:
    """Convert a messy numeric string to a float, or return None if it cannot be trusted."""
    if value is None:
        return None

    # Clean up whitespace and common non-numeric fluff
    cleaned = value.strip().upper()

    # Drop currency signs and abbreviations (e.g., 'KES', 'KSH', '$')
    cleaned = (
        cleaned.replace("KES", "")
        .strip()
    )

    # Remove commas used as thousands separators (e.g., "1,200.50" -> "1200.50")
    cleaned = cleaned.replace(",", "")

    # Handle explicit placeholders or empty fields
    if cleaned in ("", "N/A", "NONE", "NULL", "-"):
        return None

    try:
        return float(cleaned)
    except ValueError:
        # String contains untrustworthy characters (e.g., "twelve", "120A")
        return None


print(parse_number("  KES 1,200.50  "))  # Should return 1200.50
print(parse_number("    N/A"))  # Should return None




1200.5
None


In [27]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict[str, str]]]:
    """Convert quantity and unit_price_kes in valid rows; log reasons for skipped rows."""
    converted_rows = []
    skipped_rows = []

    for idx, row in enumerate(rows):
        # Read target columns
        raw_qty = row.get("quantity")
        raw_price = row.get("unit_price_kes")

        qty = parse_number(raw_qty)
        price = parse_number(raw_price)

        # Validation checks
        if qty is None:
            skipped_rows.append(
                {
                    "row_index": idx,
                    "sale_id": row.get("sale_id", "Unknown"),
                    "reason": f"Invalid or missing 'quantity' value: {repr(raw_qty)}",
                }
            )
            continue

        if price is None:
            skipped_rows.append(
                {
                    "row_index": idx,
                    "sale_id": row.get("sale_id", "Unknown"),
                    "reason": f"Invalid or missing 'unit_price_kes' value: {repr(raw_price)}",
                }
            )
            continue

        # Prevent illogical values like negative numbers
        if qty <= 0 or price <= 0:
            skipped_rows.append(
                {
                    "row_index": idx,
                    "sale_id": row.get("sale_id", "Unknown"),
                    "reason": f"Negative numeric values are not allowed (qty: {qty}, price: {price})",
                }
            )
            continue

        # Build converted record
        new_row = row.copy()
        new_row["quantity"] = int(qty)  # Quantities should be integers
        new_row["unit_price_kes"] = price
        converted_rows.append(new_row)

    return converted_rows, skipped_rows




print(f"Converted rows: {len(converted)}")
print(f"Skipped rows: {len(skipped)}")


Converted rows: 46
Skipped rows: 4
